# Stage 0b: optimized-emulator (outer-loop) hyperparameter search results

Companion notebook for `scripts/0b_hyperparameter_retune.py`. Presents that script's
already-completed hyperparameter search - cheap-search top candidates, the full
validate-phase results, and the final winning config - as clean tables for easy transfer
into the manuscript's supplemental materials (`supplement.tex`).

**Structure**: one section per agent group. **CO2-only is filled in now** (Stage 0b/0c
complete for CO2). CH4/N2O/Sulfur/BC and the multi-agent case each get their own
independent Stage 0b cheap-search + validate pass later (per `REVISIONS.md`: do **not**
reuse CO2-only's winning hyperparameters or search ranges for another agent) - those
sections are placeholders below, to be filled in as each rerun completes.

In [1]:
import os, sys
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import json
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

## CO2-only

### Table 1 - cheap-search top candidates

Top-5 (of 100 sampled) shared-config candidates by cheap-search weighted score (reduced
`num_updates=300`, single seed=0 per group; see `scripts/0b_hyperparameter_retune.py`'s
`select_candidates`). These are the candidates carried forward into the full validate
phase (Table 2).

In [2]:
HP_DIR = os.path.join(PROJECT_ROOT, "data", "SI_results", "hp_retune")

top_candidates = json.load(open(os.path.join(HP_DIR, "top_candidates_unified.json")))
rows = []
for c in top_candidates:
    row = {"config_idx": c["config_idx"], "cheap_weighted_score": c["cheap_weighted_score"]}
    row.update(c["config"])
    rows.append(row)

df_cheap = pd.DataFrame(rows).sort_values("cheap_weighted_score").reset_index(drop=True)
df_cheap

,config_idx,cheap_weighted_score,step_size,momentum,nesterov,K_inner,lr_inner,wd_inner,smoothness_weight,batch_size
0,56,0.052023,5893.589737,0.85,False,400,0.037242,0.00,0.000003,NaN
1,33,0.054359,3603.471820,0.90,True,400,0.026874,0.00,0.000000,NaN
2,63,0.057360,3989.634073,0.90,True,400,0.022166,0.01,0.000001,64.0
3,79,0.060266,2561.839090,0.85,True,400,0.025230,0.03,0.000001,NaN
4,80,0.062312,3518.011870,0.85,False,400,0.036351,0.03,0.000001,NaN


### Table 2 - validate-phase results (full `num_updates=1000`, 10 seeds/group)

For each of Table 1's candidates: mean score per group across all requested seeds, the
overall weighted-mean validate score (weights matching Fig 4's scenario-count convention:
H-ext=1, tier1=7, tier2=5, DECK=2, CS3=2, all=17), and `eligible` - `True` only if every
group has every requested seed present and stable (exactly `scripts/0b_hyperparameter_
retune.py`'s `finalize()` selection rule - reproduced here directly from the raw per-run
CSV, not re-derived by assumption).

In [3]:
GROUP_WEIGHTS = {"H-ext": 1, "tier1": 7, "tier2": 5, "DECK": 2, "CS3": 2, "all": 17}

df_val = pd.read_csv(os.path.join(HP_DIR, "validate_unified.csv"))
df_val["stable"] = df_val["stable"].astype(str) == "True"
n_seeds_expected = df_val["seed"].nunique()

group_means = df_val.groupby(["cand_idx", "group"]).agg(
    mean_score=("score", "mean"), n_seeds=("seed", "count"), all_stable=("stable", "all"),
).reset_index()

val_rows = []
for cand_idx, g in group_means.groupby("cand_idx"):
    g = g.set_index("group")
    row = {"cand_idx": int(cand_idx)}
    eligible = True
    for grp in GROUP_WEIGHTS:
        if grp in g.index:
            row[grp] = g.loc[grp, "mean_score"]
            if g.loc[grp, "n_seeds"] < n_seeds_expected or not g.loc[grp, "all_stable"]:
                eligible = False
        else:
            row[grp] = np.nan
            eligible = False
    row["eligible"] = eligible
    row["weighted_validate_score"] = (
        sum(GROUP_WEIGHTS[grp] * row[grp] for grp in GROUP_WEIGHTS) / sum(GROUP_WEIGHTS.values())
        if eligible else np.nan
    )
    val_rows.append(row)

df_validated = pd.DataFrame(val_rows).sort_values(
    "weighted_validate_score", na_position="last"
).reset_index(drop=True)
df_validated

,cand_idx,H-ext,tier1,tier2,DECK,CS3,all,eligible,weighted_validate_score
0,3,0.008976,0.042323,0.077333,0.033129,0.012641,0.069507,True,0.057796
1,0,0.003695,0.036344,0.059171,0.017410,0.012041,0.055915,False,NaN
2,1,0.003524,0.029791,0.060717,0.013967,0.007147,0.054808,False,NaN
3,2,0.006023,0.027434,0.059489,0.022088,0.006553,0.066806,False,NaN
4,4,0.005827,0.035034,0.086093,0.025034,0.020606,0.064341,False,NaN


### Table 3 - winning config

The single overall winner (`data/SI_results/hp_retune/best_config_unified.json`), used
identically across all 6 groups (H-ext/tier1/tier2/DECK/CS3/all) and consumed directly by
`scripts/0c_regenerate_checkpoints_co2.py` when regenerating checkpoints.

In [4]:
best = json.load(open(os.path.join(HP_DIR, "best_config_unified.json")))

df_winner_config = pd.DataFrame([{"cand_idx": best["cand_idx"],
                                   "weighted_validate_score": best["weighted_validate_score"],
                                   **best["config"]}])
df_winner_breakdown = pd.DataFrame([best["per_group_mean_score"]])

print("Winning config:")
display(df_winner_config)
print("Per-group validate score breakdown:")
display(df_winner_breakdown)

Winning config:


,cand_idx,weighted_validate_score,step_size,momentum,nesterov,K_inner,lr_inner,wd_inner,smoothness_weight,batch_size
0,3,0.057796,2561.83909,0.85,True,400,0.02523,0.03,0.000001,None


Per-group validate score breakdown:


,H-ext,tier1,tier2,DECK,CS3,all
0,0.008976,0.042323,0.077333,0.033129,0.012641,0.069507


### Export (for direct transfer into the supplement)

In [5]:
df_cheap.to_csv(os.path.join(HP_DIR, "co2_only_cheap_search_top5_table.csv"), index=False)
df_validated.to_csv(os.path.join(HP_DIR, "co2_only_validate_candidates_table.csv"), index=False)
df_winner_config.to_csv(os.path.join(HP_DIR, "co2_only_winner_config_table.csv"), index=False)
df_winner_breakdown.to_csv(os.path.join(HP_DIR, "co2_only_winner_breakdown_table.csv"), index=False)
print("Wrote 4 CSVs to", HP_DIR)

Wrote 4 CSVs to /Users/chriswomack/Documents/PhD/Project 2/data/SI_results/hp_retune


## CH4-only, N2O-only, Sulfur-only, BC-only, Multi-agent

Placeholder - each gets its own independent Stage 0b cheap-search + validate pass (own
search ranges re-centered on that agent's own current defaults, per `REVISIONS.md`'s
explicit caution not to transfer CO2-only's hyperparameters/ranges to other agents). Fill
in a section per group here as each rerun completes, following the CO2-only section above
as the template (same three tables + CSV export, pointed at that agent's own
`data/SI_results/hp_retune/` output files).